In [2]:
import pandas as pd
import numpy as np
import requests
import json
import paramiko
from bs4 import BeautifulSoup
from lxml import html
from openpyxl import Workbook
from openpyxl.styles import PatternFill
from datetime import datetime, timedelta
from datetime import date
import warnings
import re
warnings.filterwarnings("ignore")
import os
from pathlib import Path

In [3]:
'''# CHANGE!!! Enter file path of the Excel Source
file_source = "C:\\Users\\Pawandeep Kaur\\RepuTex\\Company - Documents\\10 Python\\Codes\\REMO Viewer\\Q1 REMO Datasets EIQ 042026.xlsx"

# Path where json and html files are saved.
JS_filepath = "C:\\Users\\Pawandeep Kaur\\RepuTex\\Company - Documents\\10 Python\\Codes\\REMO Viewer\\JSON\\"
HTML_filepath = "C:\\Users\\Pawandeep Kaur\\RepuTex\\Company - Documents\\10 Python\\Codes\\REMO Viewer\\HTML\\"
output_dir = JS_filepath '''



'# CHANGE!!! Enter file path of the Excel Source\nfile_source = "C:\\Users\\Pawandeep Kaur\\RepuTex\\Company - Documents\\10 Python\\Codes\\REMO Viewer\\Q1 REMO Datasets EIQ 042026.xlsx"\n\n# Path where json and html files are saved.\nJS_filepath = "C:\\Users\\Pawandeep Kaur\\RepuTex\\Company - Documents\\10 Python\\Codes\\REMO Viewer\\JSON\\"\nHTML_filepath = "C:\\Users\\Pawandeep Kaur\\RepuTex\\Company - Documents\\10 Python\\Codes\\REMO Viewer\\HTML\\"\noutput_dir = JS_filepath '

In [ ]:
HOME_BASE = Path.home() / "RepuTex" / "Company - Documents" / "10 Python" / "Codes" / "Electricity viewer"
JS_filepath = Path.home() / "RepuTex" / "Company - Documents" / "10 Python" / "Codes" / "Electricity viewer"/"JSON_files"
file_source  = Path.home() / "RepuTex" / "Company - Documents" / "10 Python" / "Codes" / "Electricity viewer" / "Q1 REMO Datasets EIQ 042026.xlsx"
HTML_filepath = JS_filepath = Path.home() / "RepuTex" / "Company - Documents" / "10 Python" / "Codes" / "Electricity viewer"/"HTML"
output_dir = JS_filepath
print("Input :", file_source)
print("output folder:", output_dir)

Input : C:\Users\Pawandeep Kaur\RepuTex\Company - Documents\10 Python\Codes\Electricity viewer\Q1 REMO Datasets EIQ 042026.xlsx
output folder: C:\Users\Pawandeep Kaur\RepuTex\Company - Documents\10 Python\Codes\Electricity viewer\HTML


### we currently have medium forecast only

### Certificate supply

In [52]:
gen_by_source_df = pd.read_excel(file_source, sheet_name="Gen by Source", header=0)
gen_by_source_df = gen_by_source_df.rename(columns={gen_by_source_df.columns[0]: 'Year'})
gen_by_source_df.head(3)

,Year,Supply Scenario,Distributed PV,Wind (Onshore),Solar (Utility),Hydro,Other renewable fuels,Wind (offshore),Total Renewables
0,2025,Current Pathway,32.289282,15.652980,13.117890,6.88512,0.132,0.0,68.077272
1,2026,Current Pathway,34.789282,17.836780,14.051890,6.88512,0.132,0.0,73.695072
2,2027,Current Pathway,37.289282,18.773591,15.913612,6.88512,0.132,0.0,78.993605


In [53]:
# write example
output_path = os.path.join(output_dir, "gen_cert_supply.js")

# 3) dump your data
gen_map = gen_by_source_df.to_dict(orient="records")
with open(output_path, "w", encoding="utf-8") as f:
    f.write("const genCertSupply = ")
    json.dump(gen_map, f, ensure_ascii=False, indent=2)
    f.write(";")

In [54]:
rec_by_source_df = pd.read_excel(file_source, sheet_name="REC Supply By Source", header=1)
rec_by_source_df = rec_by_source_df.rename(columns={rec_by_source_df.columns[0]: 'Year'})
rec_by_source_df.head(3)

,Year,Supply Scenario,Solar (Utility),Wind (Onshore),Wind (Offshore),Hydro,Other renewable fuels,Solar (Rooftop),Total RECs
0,2025,Current Pathway,18.714000,38.398000,0.0,1.400115,0.504000,0.0,59.016115
1,2026,Current Pathway,20.177816,43.065549,0.0,1.548549,0.479554,0.0,65.271468
2,2027,Current Pathway,22.295085,46.325583,0.0,1.696982,0.455107,0.0,70.772757


In [55]:
output_path = os.path.join(output_dir, "rec_cert_supply.js")

rec_map = rec_by_source_df.to_dict(orient="records")
with open(output_path, "w", encoding="utf-8") as f:
    f.write("const recCertSupply = ")
    json.dump(rec_map, f, ensure_ascii=False, indent=2)
    f.write(";")

### Certificate DEmand

In [56]:

demand_by_source_df = pd.read_excel(file_source, sheet_name="Vol Demand by Source", header=0)
demand_by_source_df = demand_by_source_df.rename(columns={demand_by_source_df.columns[0]: 'Year'})
demand_by_source_df.head(3)

,Year,Demand Scenario,Corporate commitments,Data Centres & Technology,Green hydrogen,Government Commitments,Green Power,Other surrenders,Total surrenders
0,2025,Current Pathway,7.4,0.376024,0.000000,4.0,2.7,0.8,15.276024
1,2026,Current Pathway,9.7,2.293083,0.129576,4.0,2.8,0.8,19.722659
2,2027,Current Pathway,10.0,3.112787,0.140603,4.1,2.9,0.8,21.053390


In [57]:
output_path = os.path.join(output_dir, "source_cert_demand.js")

source_map = demand_by_source_df.to_dict(orient="records")
with open(output_path, "w", encoding="utf-8") as f:
    f.write("const sourceCertDemand = ")
    json.dump(source_map, f, ensure_ascii=False, indent=2)
    f.write(";")

In [58]:

demand_by_ind_df = pd.read_excel(file_source, sheet_name="Vol Demand by Industry", header=0)
demand_by_ind_df = demand_by_ind_df.rename(columns={demand_by_ind_df.columns[0]: 'Year'})
demand_by_ind_df.head(3)

,Year,Demand Scenario,Accommodation & Hospitality,Agriculture,Aluminium & Alumina,Chemicals & Petrochemicals,Coal,Construction,Education,Energy Retail & Markets,...,Water Utilities,Wholesale Trade,Oil & Gas Upstream,Forest & Timber Products,Cement & Building Materials,Other Manufacturing,Government & Public Services,Green hydrogen,Data Centres & Technology,New additions
0,2025,Current Pathway,0.003682,0.008086,0.000150,0.000488,0.307895,0.047834,0.599398,1.804917,...,0.010905,0.009167,0.0,0.0,0.0,0.0,0.0,0.000000,0.418149,0.0
1,2026,Current Pathway,0.003871,0.008502,0.000182,0.000516,0.326737,0.055920,0.968684,2.073728,...,0.013428,0.009639,0.0,0.0,0.0,0.0,0.0,0.129576,2.791948,0.0
2,2027,Current Pathway,0.004026,0.008840,0.000200,0.000540,0.339630,0.057284,0.989883,2.144747,...,0.013793,0.010022,0.0,0.0,0.0,0.0,0.0,0.140603,3.652938,0.0


In [59]:
output_path = os.path.join(output_dir, "ind_cert_demand.js")

ind_map = demand_by_ind_df.to_dict(orient="records")
with open(output_path, "w", encoding="utf-8") as f:
    f.write("const indCertDemand = ")
    json.dump(ind_map, f, ensure_ascii=False, indent=2)
    f.write(";")

### Certificate price forecast

In [60]:

price_df = pd.read_excel(file_source, sheet_name="Prices", header=0)

price_df.head(3)

,Price forecast,Supply Scenario,Demand Scenario,Price
0,CAL 25,Current Pathway,Current Pathway,2.800000
1,CAL 26,Current Pathway,Current Pathway,1.684071
2,CAL 27,Current Pathway,Current Pathway,1.231491


In [61]:
output_path = os.path.join(output_dir, "price_forecast_remo.js")

price_map = price_df.to_dict(orient="records")
with open(output_path, "w", encoding="utf-8") as f:
    f.write("const priceRemo = ")
    json.dump(price_map, f, ensure_ascii=False, indent=2)
    f.write(";")

In [62]:
market_df = pd.read_excel(file_source, sheet_name="Market Balance", header=0)
market_df = market_df.rename(columns={market_df.columns[0]: 'Year'})
market_df.head(3)

,Year,Supply Scenario,Demand Scenario,REC Supply,Demand,Market Balance
0,2025,Current Pathway,Current Pathway,59.016115,48.998333,25.917782
1,2026,Current Pathway,Current Pathway,65.271468,53.510159,37.679091
2,2027,Current Pathway,Current Pathway,70.772757,54.813748,53.638100


In [63]:
output_path = os.path.join(output_dir, "market_balance_remo.js")

market_map = market_df.to_dict(orient="records")
with open(output_path, "w", encoding="utf-8") as f:
    f.write("const marketRemo = ")
    json.dump(market_map, f, ensure_ascii=False, indent=2)
    f.write(";")

### write to winscp

In [64]:
gen_source_filename = 'gen_cert_supply.js'
rec_source_filename = 'rec_cert_supply.js'
source_demand_filename ='source_cert_demand.js'
ind_demand_filename ='ind_cert_demand.js'
price_filename = 'price_forecast_remo.js'
market_filename = 'market_balance_remo.js'

# 2. SFTP parameters
#old staging site
'''hostname=  'reputex2021stg.sftp.wpengine.com'
username=  'reputex2021stg-pawandeep'
password= '#}V{Wh#sZ:j_Rf1'
port= 2222'''

#live site
hostname = 'reputex2021.sftp.wpengine.com'
port = 2222  # Default SFTP port is 2222
username = 'reputex2021-mildred'
password = 'Rc&A933[j1M5'

local_file_paths = [JS_filepath + gen_source_filename,
                    JS_filepath + rec_source_filename,
                    JS_filepath + source_demand_filename,
                    JS_filepath + ind_demand_filename,
                    JS_filepath + price_filename,
                    JS_filepath + market_filename
                    ]
remote_file_paths = ['/wp-content/uploads/mildred-json/' + gen_source_filename,
                     '/wp-content/uploads/mildred-json/' + rec_source_filename,
                     '/wp-content/uploads/mildred-json/' + source_demand_filename,
                     '/wp-content/uploads/mildred-json/' + ind_demand_filename,
                     '/wp-content/uploads/mildred-json/' + price_filename,
                     '/wp-content/uploads/mildred-json/' + market_filename
                     ]

# 3. Connect and upload
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

try:
    ssh.connect(hostname, port, username, password)
    sftp = ssh.open_sftp()

    for local_path, remote_path in zip(local_file_paths, remote_file_paths):
        sftp.put(local_path, remote_path)
        print(f"Uploaded {local_path} ➔ {remote_path}")

finally:
    if 'sftp' in locals():
        sftp.close()
    ssh.close()

print("upload complete.")

Uploaded C:\Users\Pawandeep Kaur\RepuTex\Company - Documents\10 Python\Codes\REMO Viewer\JSON\gen_cert_supply.js ➔ /wp-content/uploads/mildred-json/gen_cert_supply.js
Uploaded C:\Users\Pawandeep Kaur\RepuTex\Company - Documents\10 Python\Codes\REMO Viewer\JSON\rec_cert_supply.js ➔ /wp-content/uploads/mildred-json/rec_cert_supply.js
Uploaded C:\Users\Pawandeep Kaur\RepuTex\Company - Documents\10 Python\Codes\REMO Viewer\JSON\source_cert_demand.js ➔ /wp-content/uploads/mildred-json/source_cert_demand.js
Uploaded C:\Users\Pawandeep Kaur\RepuTex\Company - Documents\10 Python\Codes\REMO Viewer\JSON\ind_cert_demand.js ➔ /wp-content/uploads/mildred-json/ind_cert_demand.js
Uploaded C:\Users\Pawandeep Kaur\RepuTex\Company - Documents\10 Python\Codes\REMO Viewer\JSON\price_forecast_remo.js ➔ /wp-content/uploads/mildred-json/price_forecast_remo.js
Uploaded C:\Users\Pawandeep Kaur\RepuTex\Company - Documents\10 Python\Codes\REMO Viewer\JSON\market_balance_remo.js ➔ /wp-content/uploads/mildred-json